<a href="https://colab.research.google.com/github/nabinjoshi54/lis5693/blob/main/lab-6/Lab_6_new.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Lab6: Text Networks

In this lab, I am going to use a subset of my lab-5 dataset on batteries and space-related research articles to build and analyze a text network. I will preprocess the text, will construct a document-term network, identify important nodes using degree and BiRank, create a term-term projection network, examine betweenness, and analyze clusters to identify themes in the corpus.


Task 1: Load and Prepare the Dataset
For this task, I reused the dataset from lab-5 and kept only 20 articles, as instructed in canvas by profesor Lamba. I combined the `Title` and `Abstract` columns into a single text field so that each document includes both a short summary and a more detailed description. This combined text is used to create the corpus for text network analysis.

In [ ]:
!pip install textnets -q

In [ ]:
import textnets as tn
import pandas as pd
import matplotlib.pyplot as plt
import re
import string
import nltk

from nltk.corpus import stopwords
nltk.download("stopwords")

In [ ]:
tn.params["seed"] = 42

In [ ]:
url = "https://raw.githubusercontent.com/nabinjoshi54/lis5693/refs/heads/main/lab-6/lab5-batteries-and-space.csv"
df = pd.read_csv(url)

print("Original dataset shape:", df.shape)
df.head()

In [ ]:
# Keep only 20 articles
df = df.iloc[:20].copy()

print("Subset shape:", df.shape)
df.head()

In [ ]:
# Combine title and abstract
df["text"] = df["Title"].fillna("") + " " + df["Abstract"].fillna("")
df = df[["Title", "Abstract", "text"]].copy()

df.head()

Task2: Text Preprocessing and Network Size
In this step, I will clean the text by converting it to lowercase, removing punctuation, numbers, and stopwords, and keeping only meaningful alphabetic words. I will also add a small set of custom stopwords to remove general academic words that appear often but do not add much meaning. After preprocessing, I am building the document-term network and counted the number of nodes and edges in the dataset.

In [ ]:
stop_words = set(stopwords.words("english"))

custom_stopwords = {
    "et", "al", "using", "used", "use", "based", "study", "results",
    "method", "methods", "analysis", "research", "paper", "article",
    "also", "within", "however", "can", "may", "one", "two", "new",
    "show", "shown", "different"
}

stop_words = stop_words.union(custom_stopwords)

In [ ]:
def clean_doc(doc):
    doc = str(doc).lower()
    doc = re.sub(r"\d+", " ", doc)
    doc = re.sub(rf"[{re.escape(string.punctuation)}]", " ", doc)
    doc = re.sub(r"\s+", " ", doc).strip()
    tokens = doc.split()
    tokens = [w for w in tokens if w not in stop_words and len(w) > 2 and w.isalpha()]
    return " ".join(tokens)

In [ ]:
df["clean_text"] = df["text"].apply(clean_doc)
df = df[df["clean_text"].str.strip() != ""].copy()

df[["Title", "clean_text"]].head()

In [ ]:
# Create corpus
corpus = tn.Corpus(df["clean_text"], lang="en")
corpus

In [ ]:
# Build text network
t = tn.Textnet(corpus.tokenized(), min_docs=2)
t

In [ ]:
print("Number of nodes:", len(t.nodes))
print("Number of edges:", len(t.edges))

In [ ]:
nodes_df = pd.DataFrame({
    attr: t.nodes[attr] for attr in t.nodes.attributes()
})
nodes_df.head()

In [ ]:
edges_df = pd.DataFrame({
    attr: t.edges[attr] for attr in t.edges.attributes()
})
edges_df.head()
edges_df.sort_values("weight", ascending=False).head(10)


After preprocessing and building the bipartite text network, the dataset contains 126 terms and 392 edges. The term nodes represent important words extracted from the articles, while the edges represent links between documents and the terms they contain.

Task_3A: Node Types

The text network is bipartite, which means that it contains two different types of nodes: document nodes and term nodes. Document nodes represent the selected research articles. Term nodes represent the important words extracted from the corpus. An edge connects a document to a term when that term appears in the document.

In [ ]:
t.plot(
    label_nodes=True,
    show_clusters=True,
    scale_nodes_by="birank",
    scale_edges_by="weight"
)
plt.show()

In [ ]:
plt.figure(figsize=(14, 12))
t.plot(
    label_nodes=True,
    show_clusters=True,
    scale_nodes_by="birank",
    scale_edges_by="weight"
)
plt.savefig("document_term_network.png", dpi=300, bbox_inches="tight")
plt.show()

Task 3B: Degree and BiRank
Degree measures how many connections a node has. Terms with high degree appear in many documents, making them common across the dataset. BiRank goes a step further by identifying important nodes based on both their connections and the importance of the nodes they connect to. This helps highlight terms that are not only frequent but also influential in the network.

In [ ]:
t.nodes.attributes()

In [ ]:
import pandas as pd

# Convert nodes to DataFrame
nodes_df = pd.DataFrame({
    attr: t.nodes[attr] for attr in t.nodes.attributes()
})

# Add degree
nodes_df["degree"] = t.graph.degree()

# Add BiRank directly
nodes_df["birank"] = t.birank

# Now get top nodes by degree
top_degree = nodes_df.sort_values("degree", ascending=False).head(10)
top_degree


In [ ]:
top_birank = nodes_df.sort_values("birank", ascending=False).head(10)
top_birank

The top nodes by degree are the most connected nodes in the network. They appear broadly across the corpus. The top nodes by BiRank are especially important because they are connected to other important nodes and appear frequently and connect multiple documents. In this dataset, these highly ranked terms help reveal the major concepts discussed across the selected battery and space-related articles. To better understand the most important concepts in the dataset, I filtered the nodes to include only term nodes.

Task_3C: Term-Term Projection Network
The term-term projection network connects terms that appear together in the same documents. This provides a clearer view of how concepts are related to each other. Instead of focusing on documents, this network highlights relationships between words. This helps reveal the main themes and patterns in the corpus.

In [ ]:
words = t.project(node_type=tn.TERM)
# Keep only strong connections
# words = words.threshold(2)   # try 2 or 3
words

In [ ]:
plt.figure(figsize=(14, 12))
words.plot(
    label_nodes=True,
    node_label_filter=lambda n: n.degree() > 5
)
#plt.figure(figsize=(12, 10))
#words.plot(label_nodes=True, show_clusters=True)
#plt.xlim(-0.6, 0.6)
#plt.ylim(-0.6, 0.6)
#plt.savefig("term_term_network.png", dpi=300, bbox_inches="tight")
plt.show()

Task_3D:
The term-term network represents co-occurrence relationships between words. If two terms are connected, it means they frequently appear together in the same documents. This helps identify which concepts are closely related and provides insight into the overall structure of the dataset.

In [ ]:
plt.figure(figsize=(14, 12))
words.plot(
    label_nodes=True,
    show_clusters=True,
    scale_nodes_by="degree"
)
plt.savefig("term_term_network.png", dpi=300, bbox_inches="tight")
plt.show()

Task_3E: Strongly Connected Term Pairs
To identify strongly connected term pairs, I examined the edges with the highest weights in the term-term network. These edges represent pairs of terms that frequently appear together in the same documents.

In [ ]:
import pandas as pd

# Build the term-term projection network
words = t.project(node_type=tn.TERM)

# Convert term-term edges to a DataFrame
words_edges_df = pd.DataFrame({
    attr: words.edges[attr] for attr in words.edges.attributes()
})

# Add source and target node indices
words_edges_df["source"] = [e.source for e in words.edges]
words_edges_df["target"] = [e.target for e in words.edges]

# Convert projected-network nodes to a DataFrame
words_nodes_df = pd.DataFrame({
    attr: words.nodes[attr] for attr in words.nodes.attributes()
})

# Use the 'id' column as the term label
term_names = words_nodes_df["id"].tolist()

# Map source and target indices to actual term names
words_edges_df["source_term"] = words_edges_df["source"].map(lambda i: term_names[i])
words_edges_df["target_term"] = words_edges_df["target"].map(lambda i: term_names[i])

# Sort by edge weight to find the strongest term pairs
top_term_pairs = words_edges_df.sort_values("weight", ascending=False).head(10)

# Show the top term pairs
top_term_pairs[["source_term", "target_term", "weight"]]

Three strongly connected term pairs in the network are:

1. control – voltage
2. burn – pack
3. array – solar

These term pairs are strongly connected because they frequently appear together in the same documents.

Task 3G:
Betweenness centrality identifies nodes that act as bridges between different parts of the network. Terms with high betweenness are important because they connect multiple themes or clusters, helping show how different ideas in the corpus are linked together.

In [ ]:
words.top_betweenness()

In [ ]:
papers = t.project(node_type=tn.DOC)
papers.top_betweenness()

The terms with the highest betweenness centrality act as bridges across different thematic areas in the corpus. These terms are useful because they connect multiple clusters and show how broader ideas link specific concepts together.

Task4: Cluster Analysis
Clusters in the term-term network represent groups of closely related terms. These clusters can be interpreted as themes in the corpus. By examining the most important terms in each cluster, I can identify the main topics discussed in the selected articles.

In [ ]:
terms = t.project(node_type=tn.TERM, connected=True)
terms.top_cluster_nodes()

The network detected 4 clusters.

1. Cluster 0 appears to represent battery storage and cell-related concepts because it includes terms such as storage, active, application, and cell.
2. Cluster 1 appears to represent solar power systems and control because it includes terms such as solar, control, voltage, and temperature.
3. Cluster 2 appears to represent electric energy systems and performance because it includes terms such as energy, electric, pack, and performance.
4. Cluster 3 appears to represent satellite operation and thermal behavior because it includes terms such as operation, satellite, thermal, and system.

One important thing form text modeling is that the clusters detected in the text network are similar to the topics found in lab-5 topic modeling. Both methods highlight themes related to battery materials, electrochemical performance, energy etc. However, text network analysis provides a clearer view of how specific terms are directly connected, while topic modeling groups words into broader topic distributions.

Task5: Save Graphs

I saved the main graph visualizations so they can be uploaded to GitHub along with the notebook. These figures provide visual evidence of the document-term network and the term-term projection network used in the analysis.

In [ ]:
# Save graph object
words.save_graph("term_network.gml")

In [ ]:
# Save bipartite network image
plt.figure(figsize=(14, 12))
t.plot(
    label_nodes=True,
    show_clusters=True,
    scale_nodes_by="birank",
    scale_edges_by="weight"
)
plt.savefig("document_term_network.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Save term-term network image
plt.figure(figsize=(14, 12))
words.plot(label_nodes=True, show_clusters=True)
plt.savefig("term_term_network.png", dpi=300, bbox_inches="tight")
plt.show()

Summary Tables
The following tables summarize the most important nodes and relationships in the text network.

In [ ]:
print("Top nodes by degree")
display(top_degree)

print("Top nodes by BiRank")
display(top_birank)

print("Top term pairs by weight")
display(top_term_pairs)

Reflection:
One thing that went well in this lab was understanding how text networks reveal relationships between terms and documents. The term-term network was especially helpful in showing which concepts frequently appear together. Initially, the graph was very crowded because even 15–20 articles produced hundreds of terms, making it hard to read.

To better understand the structure, I am tempted to experiment by reducing the number of articles. This made the network much clearer and helped me visually see how terms are connected. Since the assignment required 15-20 topics, I had done this in another lab file and will submit it as a labwork. However, this experiment helped me better interpret the results and understand the behavior of text networks.

One challenge was working with the `textnets` library. I have never heard about it before.

In the future, text network analysis could be useful in my research to explore relationships between key concepts in scientific literature, especially in areas like battery materials, performance, their application in aerospace and space systems, adn degradation. It can complement topic modeling by showing how ideas are directly connected.